# Assignment 2 - Question 1: Used Cars

Data: `data/train.csv` (used car listings). Each row is one car.
Target: `Price`, in lakhs (1 lakh = 100,000 rupees).

Parts: (a) missing values, (b) remove units, (c) one-hot encoding, (d) new feature, (e) select/filter/rename/mutate/arrange/group by.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

DATA_PATH = "../data/train.csv"

df = pd.read_csv(DATA_PATH)
# The first column is an unnamed row ID from the original file; use it as the index.
if df.columns[0].startswith("Unnamed") or df.columns[0] == "":
    df = df.set_index(df.columns[0])
df.index.name = "Listing_ID"

print("Shape:", df.shape)
df.head()

Shape: (5847, 13)


,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
Listing_ID,,,,,,,,,,,,,
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


## (a) Missing values

First I count the missing values (NaN) in every column.

In [2]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
})
missing[missing.missing_count > 0].sort_values("missing_count", ascending=False)

,missing_count,missing_pct
New_Price,5032,86.06
Seats,38,0.65
Power,36,0.62
Engine,36,0.62
Mileage,2,0.03


Some missing values are hidden as text, so `isna()` doesn't catch them:

- `0.0 kmpl` in Mileage. A car can't get 0 km per litre, so this means missing.
- 3 Mileage values are in `km/kg`. That unit is for CNG cars, but these cars are petrol/diesel, and km/kg can't be converted to kmpl. I set them to missing. Later, 2 of them get filled from the same car model and the third gets dropped (no matching model).

I change these to NaN before handling the missing values.

In [3]:
print("Mileage == '0.0 kmpl':", (df["Mileage"] == "0.0 kmpl").sum())
print("Mileage in km/kg:     ", df["Mileage"].str.endswith("km/kg", na=False).sum())

df["Mileage"] = df["Mileage"].replace("0.0 kmpl", np.nan)
df.loc[df["Mileage"].str.endswith("km/kg", na=False), "Mileage"] = np.nan

missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
})
missing = missing[missing.missing_count > 0].sort_values("missing_count", ascending=False)
missing

Mileage == '0.0 kmpl': 40
Mileage in km/kg:      3


,missing_count,missing_pct
New_Price,5032,86.06
Mileage,45,0.77
Seats,38,0.65
Power,36,0.62
Engine,36,0.62


### What I did with each column and why

- `New_Price` (about 86% missing): drop the column after part (b). The assignment lets us impute or drop. Dropping the rows would throw away 86% of the data, and filling in a mean or median would mean most of the column is made up. I keep it until part (b) so I can show the unit removal.
- `Engine`, `Power`, `Mileage` (0.6%, 0.6%, 0.8% missing): fill with the mode of the same car model (`Name`). Cars of the same model usually have the same engine, power and mileage, so this uses a real value instead of a guess. One overall mean would be wrong for most cars since power goes from about 34 to 500+ bhp. The rows with no matching model get dropped. That's only 63 rows (1.1%), including the 2 Electric cars, which have no mileage rating. The assignment says fuel type is petrol or diesel, so losing them doesn't matter.
- `Seats` (under 1% missing): fill with the mode, same model first and then the overall mode (5). Seats is a whole number, so a mean like 5.28 seats doesn't make sense.
- The other columns have no missing values.

In [4]:
def model_mode(col):
    """Most common non-missing value of `col` among listings of the same car model (Name)."""
    return df.groupby("Name")[col].transform(
        lambda x: x.mode().iloc[0] if x.notna().any() else np.nan
    )

rows_before = len(df)
report = []
for col in ["Engine", "Power", "Mileage", "Seats"]:
    before = df[col].isna().sum()
    df[col] = df[col].fillna(model_mode(col))
    report.append((col, before, before - df[col].isna().sum(), df[col].isna().sum()))

pd.DataFrame(report, columns=["column", "missing_before", "filled_from_same_model", "still_missing"])

,column,missing_before,filled_from_same_model,still_missing
0,Engine,36,5,31
1,Power,36,5,31
2,Mileage,45,5,40
3,Seats,38,6,32


In [5]:
# Seats: fill whatever is left with the overall mode (5 seats).
seats_mode = df["Seats"].mode().iloc[0]
df["Seats"] = df["Seats"].fillna(seats_mode)
print("Seats overall mode used:", seats_mode)

# Engine / Power / Mileage: drop the few rows that have no same-model reference.
df = df.dropna(subset=["Engine", "Power", "Mileage"])
dropped = rows_before - len(df)
print(f"Rows dropped: {dropped} of {rows_before} ({dropped / rows_before:.1%})")
print("Rows remaining:", len(df))

df.isna().sum()

Seats overall mode used: 5.0
Rows dropped: 63 of 5847 (1.1%)
Rows remaining: 5784


Name                    0
Location                0
Year                    0
Kilometers_Driven       0
Fuel_Type               0
Transmission            0
Owner_Type              0
Mileage                 0
Engine                  0
Power                   0
Seats                   0
New_Price            4974
Price                   0
dtype: int64

Now the only column with missing values is `New_Price`, which gets dropped after part (b).

## (b) Remove units

I strip `kmpl` from Mileage, `CC` from Engine, `bhp` from Power and `Lakh` from New_Price, and keep just the numbers. A few new prices are in crore (`Cr`, 1 Cr = 100 lakh), so I multiply those by 100 so everything is in lakhs like `Price`.

In [6]:
print("Unit labels found:")
for col in ["Mileage", "Engine", "Power", "New_Price"]:
    print(f"  {col:10s}", df[col].dropna().str.split().str[-1].value_counts().to_dict())

df[["Mileage", "Engine", "Power", "New_Price"]].dropna().head()

Unit labels found:
  Mileage    {'kmpl': 5784}
  Engine     {'CC': 5784}
  Power      {'bhp': 5784}
  New_Price  {'Lakh': 793, 'Cr': 17}


,Mileage,Engine,Power,New_Price
Listing_ID,,,,
2,18.2 kmpl,1199 CC,88.7 bhp,8.61 Lakh
7,11.36 kmpl,2755 CC,171.5 bhp,21 Lakh
10,21.56 kmpl,1462 CC,103.25 bhp,10.65 Lakh
15,13.5 kmpl,2477 CC,175.56 bhp,32.01 Lakh
20,22.69 kmpl,1995 CC,190 bhp,47.87 Lakh


In [7]:
def strip_unit(series):
    """Keep only the leading number in strings like '19.67 kmpl' and return floats."""
    return series.astype(str).str.extract(r"([\d.]+)", expand=False).astype(float)

df["Mileage"] = strip_unit(df["Mileage"])   # kmpl
df["Engine"]  = strip_unit(df["Engine"])    # CC
df["Power"]   = strip_unit(df["Power"])     # bhp

# New_Price: convert Cr to lakh so the whole column uses one unit.
new_num = strip_unit(df["New_Price"])
is_crore = df["New_Price"].str.contains("Cr", na=False)
df["New_Price"] = np.where(is_crore, new_num * 100, new_num)
print("New_Price values converted from Cr to lakh:", is_crore.sum())

df[["Mileage", "Engine", "Power", "New_Price"]].dtypes

New_Price values converted from Cr to lakh: 17


Mileage      float64
Engine       float64
Power        float64
New_Price    float64
dtype: object

In [8]:
# Cleaned values, including the New_Price rows that were converted from Cr to lakh.
print(df[["Mileage", "Engine", "Power", "New_Price"]].describe().round(2), "\n")
df.loc[is_crore, ["Name", "New_Price"]].head()

       Mileage   Engine    Power  New_Price
count  5784.00  5784.00  5784.00     810.00
mean     18.29  1628.62   113.67      23.15
std       4.11   598.96    53.81      26.27
min       6.40   624.00    34.20       3.91
25%      15.29  1198.00    78.00       8.09
50%      18.20  1496.00    98.60      11.70
75%      21.10  1991.00   138.10      26.56
max      28.40  5998.00   560.00     230.00 



,Name,New_Price
Listing_ID,,
148,Audi RS5 Coupe,128.0
327,Audi Q7 45 TDI Quattro Technology,104.0
489,Mercedes-Benz GLS 350d 4MATIC,100.0
1336,Audi Q7 45 TDI Quattro Technology,104.0
1505,Land Rover Range Rover Sport SE,139.0


All four columns are numbers now. As planned in (a), I drop `New_Price` here.

In [9]:
df = df.drop(columns=["New_Price"])
print("Missing values left in the dataset:", int(df.isna().sum().sum()))
print("Shape:", df.shape)
df.head()

Missing values left in the dataset: 0
Shape: (5784, 12)


,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price
Listing_ID,,,,,,,,,,,,
1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67,1582.0,126.20,5.0,12.50
2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,18.20,1199.0,88.70,5.0,4.50
3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77,1248.0,88.76,7.0,6.00
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.20,1968.0,140.80,5.0,17.74
6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08,1461.0,63.10,5.0,3.50


## (c) One-hot encoding

`pd.get_dummies` turns `Fuel_Type` and `Transmission` into 0/1 columns, one per category.

In [10]:
df_encoded = pd.get_dummies(df, columns=["Fuel_Type", "Transmission"], dtype=int)

ohe_cols = [c for c in df_encoded.columns if c.startswith(("Fuel_Type_", "Transmission_"))]
print("New one-hot columns:", ohe_cols)
df_encoded[["Name"] + ohe_cols].head()

New one-hot columns: ['Fuel_Type_Diesel', 'Fuel_Type_Petrol', 'Transmission_Automatic', 'Transmission_Manual']


,Name,Fuel_Type_Diesel,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual
Listing_ID,,,,,
1,Hyundai Creta 1.6 CRDi SX Option,1,0,0,1
2,Honda Jazz V,0,1,0,1
3,Maruti Ertiga VDI,1,0,0,1
4,Audi A4 New 2.0 TDI Multitronic,1,0,1,0
6,Nissan Micra Diesel XV,1,0,0,1


## (d) New feature: Car_Age

`Car_Age` = current year - `Year` (same idea as `mutate()` in R). Age is easier to read than the model year, and older cars usually sell for less. The correlation is printed below.

I fixed the current year at 2026 so the results don't change if this is run later. The newest cars in the data are 2019 models.

In [11]:
CURRENT_YEAR = 2026  # fixed (not date.today()) so the results are the same whenever this is re-run
print("Current year used:", CURRENT_YEAR)

df_encoded["Car_Age"] = CURRENT_YEAR - df_encoded["Year"]
print("Correlation between Car_Age and Price:", round(df_encoded["Car_Age"].corr(df_encoded["Price"]), 2))
df_encoded[["Name", "Year", "Car_Age", "Price"]].head()

Current year used: 2026
Correlation between Car_Age and Price: -0.3


,Name,Year,Car_Age,Price
Listing_ID,,,,
1,Hyundai Creta 1.6 CRDi SX Option,2015,11,12.50
2,Honda Jazz V,2011,15,4.50
3,Maruti Ertiga VDI,2012,14,6.00
4,Audi A4 New 2.0 TDI Multitronic,2013,13,17.74
6,Nissan Micra Diesel XV,2013,13,3.50


## (e) select, filter, rename, mutate, arrange, group by

In pandas these are: `df[[cols]]`, `query()`, `rename()`, `assign()`, `sort_values()`, and `groupby().agg()`.

### select: pick the columns I need

In [12]:
selected = df_encoded[["Name", "Location", "Car_Age", "Kilometers_Driven",
                       "Transmission_Automatic", "Power", "Mileage", "Price"]]
selected.head()

,Name,Location,Car_Age,Kilometers_Driven,Transmission_Automatic,Power,Mileage,Price
Listing_ID,,,,,,,,
1,Hyundai Creta 1.6 CRDi SX Option,Pune,11,41000,0,126.20,19.67,12.50
2,Honda Jazz V,Chennai,15,46000,0,88.70,18.20,4.50
3,Maruti Ertiga VDI,Chennai,14,87000,0,88.76,20.77,6.00
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,13,40670,1,140.80,15.20,17.74
6,Nissan Micra Diesel XV,Jaipur,13,86999,0,63.10,23.08,3.50


### filter: automatic cars 10 years old or newer

In [13]:
filtered = selected.query("Transmission_Automatic == 1 and Car_Age <= 10")
print(f"{len(filtered)} of {len(selected)} listings match")
filtered.head()

573 of 5784 listings match


,Name,Location,Car_Age,Kilometers_Driven,Transmission_Automatic,Power,Mileage,Price
Listing_ID,,,,,,,,
7,Toyota Innova Crysta 2.8 GX AT 8S,Mumbai,10,36000,1,171.50,11.36,17.50
45,Toyota Corolla Altis G AT,Mumbai,10,52001,1,138.03,14.28,10.11
74,Honda Brio VX AT,Coimbatore,10,37300,1,86.80,16.50,6.46
92,Audi A6 2011-2015 35 TDI Premium,Coimbatore,8,48367,1,174.33,17.68,39.58
110,Toyota Innova Crysta 2.8 ZX AT,Mumbai,9,30000,1,171.50,11.36,19.25


### rename: add units to the column names

In [14]:
renamed = filtered.rename(columns={
    "Kilometers_Driven": "Km_Driven",
    "Power": "Power_bhp",
    "Mileage": "Mileage_kmpl",
    "Price": "Price_lakh",
})
renamed.head()

,Name,Location,Car_Age,Km_Driven,Transmission_Automatic,Power_bhp,Mileage_kmpl,Price_lakh
Listing_ID,,,,,,,,
7,Toyota Innova Crysta 2.8 GX AT 8S,Mumbai,10,36000,1,171.50,11.36,17.50
45,Toyota Corolla Altis G AT,Mumbai,10,52001,1,138.03,14.28,10.11
74,Honda Brio VX AT,Coimbatore,10,37300,1,86.80,16.50,6.46
92,Audi A6 2011-2015 35 TDI Premium,Coimbatore,8,48367,1,174.33,17.68,39.58
110,Toyota Innova Crysta 2.8 ZX AT,Mumbai,9,30000,1,171.50,11.36,19.25


### mutate: add km driven per year

In [15]:
mutated = renamed.assign(
    Km_per_Year=lambda d: (d["Km_Driven"] / d["Car_Age"]).round(0),
)
mutated[["Name", "Car_Age", "Km_Driven", "Km_per_Year", "Price_lakh"]].head()

,Name,Car_Age,Km_Driven,Km_per_Year,Price_lakh
Listing_ID,,,,,
7,Toyota Innova Crysta 2.8 GX AT 8S,10,36000,3600.0,17.50
45,Toyota Corolla Altis G AT,10,52001,5200.0,10.11
74,Honda Brio VX AT,10,37300,3730.0,6.46
92,Audi A6 2011-2015 35 TDI Premium,8,48367,6046.0,39.58
110,Toyota Innova Crysta 2.8 ZX AT,9,30000,3333.0,19.25


### arrange: sort by price, highest first

In [16]:
arranged = mutated.sort_values("Price_lakh", ascending=False)
arranged[["Name", "Location", "Car_Age", "Km_per_Year", "Price_lakh"]].head(10)

,Name,Location,Car_Age,Km_per_Year,Price_lakh
Listing_ID,,,,,
4079,Land Rover Range Rover 3.0 Diesel LWB Vogue,Hyderabad,9,2778.0,160.00
1505,Land Rover Range Rover Sport SE,Kochi,7,3716.0,97.07
1974,BMW 7 Series 740Li,Coimbatore,8,3508.0,93.67
1984,BMW 7 Series 740Li,Bangalore,9,1941.0,93.00
2095,Mercedes-Benz SLC 43 AMG,Coimbatore,7,361.0,83.96
2422,Jaguar XJ 3.0L Portfolio,Delhi,10,1200.0,79.00
1885,Mercedes-Benz GLS 350d Grand Edition,Delhi,8,750.0,79.00
2319,Mercedes-Benz E-Class E 350 d,Coimbatore,7,1330.0,78.80
655,Jaguar XJ 2.0L Portfolio,Mumbai,9,1167.0,75.00


### group by + summarize: average price and age by city

In [17]:
city_summary = (
    df_encoded.groupby("Location")
    .agg(
        Listings=("Price", "size"),
        Avg_Price_lakh=("Price", "mean"),
        Avg_Car_Age=("Car_Age", "mean"),
    )
    .sort_values("Avg_Price_lakh", ascending=False)
    .round(2)
)
city_summary

,Listings,Avg_Price_lakh,Avg_Car_Age
Location,,,
Coimbatore,626,15.00,10.56
Bangalore,344,13.46,13.12
Kochi,635,11.29,10.46
Hyderabad,703,10.00,13.16
Delhi,533,9.91,12.62
Mumbai,755,9.54,12.61
Ahmedabad,216,8.38,12.68
Chennai,473,7.92,13.92
Pune,580,6.87,13.56
